# multiply-back — faded example 2: Implement multiply_back1 (gradient for second operand)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `multiply-back`. Running the beacon reports progress on the `Backprop: multiply_back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: multiply_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`multiply-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "multiply-back"
DD_SUBTOPIC = "Backprop: multiply_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

By symmetry of multiplication, `dL/dy = grad_out * x` — the *other* operand compared to `multiply_back0`. Then `unbroadcast(grad, y)` collapses any dimensions that were expanded when `y` was broadcast to the output shape. Writing both back functions as symmetric pairs is the standard pattern for binary operation backward implementations.

## Faded exercise 2

Implement `multiply_back1(grad_out, out, x, y)` that returns `dL/dy`.

`unbroadcast(grad, original)` is already defined.

Steps:
1. Compute `raw = grad_out * x`.
2. Return `unbroadcast(raw, y)`.

Your task: **fill in the computation of raw and the unbroadcast call**.

**Fill in:** Computing raw = grad_out * x and returning unbroadcast(raw, y) to collapse any broadcast axes back to y's original shape.

In [ ]:
import torch
from torch import Tensor

def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    raise NotImplementedError()  # TODO: Computing raw = grad_out * x and returning unbroadcast(raw, y) to collapse any broadcast axes back to y's original shape.

def _test():
    import torch
    torch.manual_seed(0)
    x = torch.randn(5, 3, requires_grad=True)
    y = torch.randn(1, 3, requires_grad=True)
    out = x * y
    out.sum().backward()
    grad_out = torch.ones(5, 3)
    gy = multiply_back1(grad_out, out.detach(), x.detach(), y.detach())
    assert gy.shape == (1, 3), f"shape: {gy.shape}"
    assert torch.allclose(gy, y.grad, atol=1e-6)


def _test():
    import torch
    torch.manual_seed(0)
    x = torch.randn(5, 3, requires_grad=True)
    y = torch.randn(1, 3, requires_grad=True)
    out = x * y
    out.sum().backward()
    gy = multiply_back1(torch.ones(5,3), out.detach(), x.detach(), y.detach())
    assert gy.shape == (1, 3), f"shape: {gy.shape}"
    assert torch.allclose(gy, y.grad, atol=1e-6)
    # No-broadcast case
    torch.manual_seed(1)
    a = torch.randn(2, 4, requires_grad=True)
    b = torch.randn(2, 4, requires_grad=True)
    (a * b).sum().backward()
    gb = multiply_back1(torch.ones(2,4), (a*b).detach(), a.detach(), b.detach())
    assert gb.shape == (2, 4)
    assert torch.allclose(gb, b.grad, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
from torch import Tensor

def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    raw = grad_out * x
    return unbroadcast(raw, y)

def _test():
    import torch
    torch.manual_seed(0)
    x = torch.randn(5, 3, requires_grad=True)
    y = torch.randn(1, 3, requires_grad=True)
    out = x * y
    out.sum().backward()
    grad_out = torch.ones(5, 3)
    gy = multiply_back1(grad_out, out.detach(), x.detach(), y.detach())
    assert gy.shape == (1, 3)
    assert torch.allclose(gy, y.grad, atol=1e-6)
```
</details>